# Data Downloading as CIF file and other property

1. Fetching crystal structures from Materials Project API

In [1]:
# Import required libraries
import os
import time
import warnings
import csv
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

from pymatgen.core.structure import Structure
from pymatgen.io.cif import CifWriter
from mp_api.client import MPRester


C:\Users\Prabhat\Documents\Semester 7\CO4015 - Mini Project\cgcnn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##  Fetch Data from Materials Project API
- Environment variable: `MP_API`

In [2]:
# Load environment variables from .env file
load_dotenv()

# Get API key from environment variable
api_key = os.getenv('MP_API')
# api_key = "your_api_key_here"  # Fallback: uncomment and replace if not using .env

if not api_key:
    warnings.warn("MP_API environment variable not set. Please provide API key directly if needed.")
else:
    print("API key loaded successfully.")


API key loaded successfully.


In [3]:
# ---- Config ----
MAX_MATERIALS = 50
CIF_DIR = "cif_structures"  # Updated to align with Notebooks 05 and 06
os.makedirs(CIF_DIR, exist_ok=True)

# Fields to retrieve
fields = [
    "material_id", "structure", "formula_pretty",
    "formation_energy_per_atom", "energy_above_hull",
    "band_gap", "density", "volume"
]

cif_count = 0
data_list = []

if api_key:
    print("Connecting to Materials Project API...")
    with MPRester(api_key) as mpr:
        # Use a tuple for ranges
        all_ids = mpr.materials.summary.search(
            energy_above_hull=(0, 0.1),
            fields=["material_id"]
        )
        material_ids = [doc.material_id for doc in all_ids]
        
        # Limit total materials to 1000 as requested
        material_ids = material_ids[:1000]
        
        # Optional: limit total materials for testing (uncomment to use)
        
        print(f"🔍 Found {len(material_ids)} candidate materials")

        # Process in batches
        for i in range(0, len(material_ids), MAX_MATERIALS):
            batch_ids = material_ids[i:i + MAX_MATERIALS]
            print(f"\n📦 Processing batch {i // MAX_MATERIALS + 1}: {len(batch_ids)} materials")

            try:
                docs = mpr.materials.summary.search(
                    material_ids=batch_ids,
                    fields=fields
                )
                print(f"✅ Retrieved {len(docs)} materials")

                for doc in tqdm(docs, desc="Saving CIFs"):
                    try:
                        # Require a structure to write CIF
                        structure = getattr(doc, 'structure', None)
                        if structure is None:
                            continue

                        cif_path = os.path.join(CIF_DIR, f"{doc.material_id}.cif")
                        CifWriter(structure).write_file(cif_path)
                        
                        # Pick a target property
                        target_prop = getattr(doc, 'formation_energy_per_atom', None)
                        if target_prop is None:
                            target_prop = getattr(doc, 'energy_above_hull', None)

                        # If no target prop, drop the CIF we just wrote
                        if target_prop is None:
                            try:
                                os.remove(cif_path)
                            except OSError:
                                pass
                            continue

                        cif_count += 1
                        data_list.append({
                            "material_id": doc.material_id,
                            "cif_file": f"{doc.material_id}.cif",
                            "formula": getattr(doc, 'formula_pretty', 'Unknown'),
                            "formation_energy_per_atom": getattr(doc, 'formation_energy_per_atom', None),
                            "energy_above_hull": getattr(doc, 'energy_above_hull', None),
                            "band_gap": getattr(doc, 'band_gap', None),
                            "density": getattr(doc, 'density', None),
                            "volume": getattr(doc, 'volume', None),
                            "target": target_prop,
                        })

                    except Exception as e:
                        print(f"⚠️ Error processing {getattr(doc, 'material_id', 'unknown')}: {e}")
                        continue

            except Exception as e:
                print(f"❌ API error: {e}")
                time.sleep(5)  # brief backoff

            time.sleep(1)  # Respect API rate limits

    # Save metadata
    df = pd.DataFrame(data_list)
    df.to_csv("cgcnn_dataset.csv", index=False)
    
    # Save id_prop.csv for backwards compatibility with notebooks 05 and 06 (header=False)
    if len(df) > 0:
        df[['material_id', 'target']].to_csv("id_prop.csv", index=False, header=False)

    print(f"\n✅ Saved {cif_count} CIF files to {CIF_DIR}/")
    print(f"✅ Metadata saved to cgcnn_dataset.csv and id_prop.csv with {len(data_list)} entries")
else:
    print("Skipping API fetch since MP_API key is not available.")

Connecting to Materials Project API...


Retrieving SummaryDoc documents: 100%|██████████████████████████████████████████████████████████████████████| 103644/103644 [00:22<00:00, 4507.23it/s]


🔍 Found 1000 candidate materials

📦 Processing batch 1: 50 materials


Retrieving SummaryDoc documents: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 365995.11it/s]


✅ Retrieved 50 materials


...


📦 Processing batch 20: 50 materials


Retrieving SummaryDoc documents: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 355449.49it/s]


✅ Retrieved 50 materials


Saving CIFs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 426.74it/s]



✅ Saved 1000 CIF files to cif_structures/
✅ Metadata saved to cgcnn_dataset.csv and id_prop.csv with 1000 entries
